# EMA Drug Comparison RAG — Portfolio Demo

This notebook demonstrates a retrieval-augmented generation (RAG) system for comparing regulatory information for **Jardiance (empagliflozin)** and **Forxiga (dapagliflozin)** using EMA Summary of Product Characteristics (SmPC) evidence.

The system is designed to produce grounded comparisons with traceable evidence rather than relying on the language model's general medical knowledge.

> **Scope:** Technical portfolio demonstration based on EMA SmPC documents; not medical advice.


## 1. Architecture

**EMA Product Information PDFs → Annex I / SmPC extraction → section-aware parsing and chunking → MiniLM embeddings → balanced per-drug candidate retrieval → cross-encoder reranking → same-section neighbor expansion → grounded GPT generation**

Key design choices:
- Preserve regulatory section and page metadata.
- Retrieve candidates independently for each drug.
- Rerank dense candidates with a cross-encoder.
- Add immediate neighbors from the same section to reduce evidence fragmentation.
- Generate answers only from supplied EMA evidence and cite evidence IDs.


## 2. Project setup

This notebook reuses the project's existing modules rather than duplicating the RAG implementation. Run it from the repository root or from the `notebooks` directory.


In [1]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")

Project root: c:\Users\gokif\projects\ema_drug_comparison_rag
Working directory: c:\Users\gokif\projects\ema_drug_comparison_rag


## 3. Corpus integrity

The current corpus contains **99 section-aware chunks**: **48 Jardiance** and **51 Forxiga**. Each chunk retains drug, active substance, SmPC section, source pages, chunk ID, and text.


In [2]:
CHUNKS_FILE = PROJECT_ROOT / "data" / "embeddings" / "chunks.json"

with open(CHUNKS_FILE, encoding = "utf-8") as f:
    chunks = json.load(f)

counts = {}
for chunk in chunks:
    counts[chunk["drug"]] = counts.get(chunk["drug"], 0) + 1

print(f"Total chunks: {len(chunks)}")
for drug, count in sorted(counts.items()):
    print(f"{drug}: {count}")

assert len(chunks) == 99
assert counts.get("Jardiance") == 48
assert counts.get("Forxiga") == 51

Total chunks: 99
Forxiga: 51
Jardiance: 48


## 4. Retrieval demonstration

Representative query:

> **Compare the ketoacidosis warnings for Jardiance and Forxiga.**

The next cell runs the current retrieval pipeline and displays selected evidence metadata. Model loading can take a short time on the first run.


In [3]:
from src.retriever import retrieve

query = "Compare the ketoacidosis warnings for Jardiance and Forxiga."

retrieved = retrieve(query)

for drug, items in retrieved["evidence"].items():
    print(f"\n{drug}")
    print("-" * 60)

    for item in items:
        print(
            f"Section {item['section']:4} | "
            f"Pages {item['pages']} | "
            f"{item['chunk_id']}"
        )

c:\Users\gokif\projects\ema_drug_comparison_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1509.75it/s]


Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 406.62it/s]



Jardiance
------------------------------------------------------------
Section 4.4  | Pages [4] | jardiance_smpc_4_4_001
Section 4.4  | Pages [4, 5] | jardiance_smpc_4_4_002
Section 4.8  | Pages [9, 10] | jardiance_smpc_4_8_002

Forxiga
------------------------------------------------------------
Section 4.4  | Pages [4, 5] | forxiga_smpc_4_4_002
Section 4.4  | Pages [4] | forxiga_smpc_4_4_001
Section 5.1  | Pages [16, 17] | forxiga_smpc_5_1_004


## 5. Evidence package

The evidence layer expands selected chunks with immediate neighboring chunks from the **same drug and same SmPC section**. This addresses cases where the correct section is retrieved but important comparison details occur in an adjacent chunk.


In [4]:
from src.evidence import build_evidence, build_context

evidence = build_evidence(query)
context = build_context(evidence)
print(context)

[jardiance_1]
Drug: Jardiance (empagliflozin)
SmPC section 4.4: Special warnings and precautions for use
Pages: 4
Chunk ID: jardiance_smpc_4_4_001

General 
 
Empagliflozin should not be used in patients with type 1 diabetes mellitus (see “Ketoacidosis” in 
section 4.4). 
 
Ketoacidosis 
 
Cases of ketoacidosis, including life-threatening and fatal cases, have been reported in patients with 
diabetes mellitus treated with SGLT2 inhibitors, including empagliflozin. In a number of cases, the 
presentation of the condition was atypical with only moderately increased blood glucose values, below 
14 mmol/L (250 mg/dL). It is not known if ketoacidosis is more likely to occur with higher doses of 
empagliflozin. Although ketoacidosis is less likely to occur in patients without diabetes mellitus, cases 
have also been reported in these patients. 
 
The risk of ketoacidosis must be considered in the event of non-specific symptoms such as nausea, 
vomiting, anorexia, abdominal pain, excessive th

## 6. Grounded answer generation

The following cell requires the API key configured for the project and incurs API usage. It is commented out by default so that opening or running the portfolio notebook does not automatically make a paid generation call.


In [5]:
# Uncomment for a live grounded comparison:
#
# from src.generator import generate_answer
# answer = generate_answer(query)
# print(answer)

## 7. Evaluation results

The project was evaluated at multiple layers rather than relying only on final-answer quality.

| Stage | Result |
|---|---:|
| Baseline paired section retrieval | 75% |
| Balanced per-drug retrieval (v2) | 95% |
| Baseline generation coverage | 76.3% |
| v2 generation coverage | 85.6% |
| Neighbor-expanded generation coverage (v3) | 87.3% |
| v3 evidence-point coverage | 93.0% |

The evidence evaluation found that **19 of 20 questions had 100% evidence-point coverage**. The remaining broad multi-intent question had **6.7% evidence coverage**.

The saved evidence-evaluation JSON can be inspected without making new API calls.


In [6]:
EVIDENCE_RESULTS = PROJECT_ROOT / "evaluation" / "evidence_results.json"

with open(EVIDENCE_RESULTS, encoding = "utf-8") as f:
    evidence_results = json.load(f)

print(f"Questions: {evidence_results['questions']}")
print(f"Overall evidence coverage: {evidence_results['overall_evidence_coverage']:.1%}")
print(
    f"Supported expected points: "
    f"{evidence_results['supported_points']}/{evidence_results['expected_points']}"
)

Questions: 20
Overall evidence coverage: 93.0%
Supported expected points: 185/199


## 8. Known limitation: broad multi-intent queries

The current system performs best when a query addresses **one focused regulatory topic at a time**.

`eval_003` combines hepatic impairment, elderly patients, paediatric patients, and method of administration. Although the relevant information exists in **SmPC section 4.2** for both medicines, semantic retrieval selected passages from other sections that were individually relevant to parts of the compound query. Evidence-point coverage was **6.7%**.

This is a retrieval limitation rather than missing source data: deterministic corpus tests confirm that section 4.2 chunks exist for both drugs.

For this version, broad multi-intent questions should be split into focused queries:
- Compare hepatic-impairment dosing recommendations.
- Compare recommendations for elderly patients.
- Compare paediatric dosing recommendations.
- Compare methods of administration.

The difficult question remains in the evaluation set as a documented failure case rather than being removed after evaluation.


## 9. What the evaluation revealed

Three distinct failure modes emerged:

**Cross-drug candidate competition.** The original global candidate pool could favor one medicine. Balanced per-drug candidate retrieval increased paired section retrieval from **75% to 95%**.

**Within-section evidence fragmentation.** A relevant section could be retrieved while an adjacent chunk contained required details. Same-section neighbor expansion improved generation coverage, particularly for difficult interaction questions.

**Multi-intent regulatory retrieval.** A broad question can resemble several SmPC sections semantically even when the regulatory answer is concentrated in one section. `eval_003` remains the principal documented example.

This layered evaluation helps distinguish retrieval failures, evidence-coverage failures, and generation failures.


## 10. Limitations and future work

- The corpus is limited to the selected EMA SmPC documents for Jardiance and Forxiga.
- The system is not a substitute for current official product information or professional medical judgment.
- Broad multi-intent queries can reduce retrieval reliability.
- Final-answer quality depends on both evidence retrieval and the generator's use of that evidence.
- The evaluation set is retained unchanged, including the known difficult query.

Possible future work includes hybrid lexical/dense retrieval, stronger regulatory-intent routing, section-level retrieval, and evaluation on additional medicines and unseen regulatory questions.


## 11. CLI usage

The same pipeline can be used through the project's command-line interface:

```powershell
python -m src.ask
```

This notebook is a portfolio walkthrough of the architecture, evidence flow, evaluation results, and known limitations.
